# Podcast Listening Time Prediction

This notebook is designed to predict the listening time of podcast episodes based on various features such as episode length, guest popularity, number of ads, and more. The workflow includes data preprocessing, feature engineering, model training, and evaluation. The final predictions are saved in a submission file for further analysis.

#### Import Libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_log_error
from sklearn.ensemble import RandomForestRegressor

#### Read train and test data

In [ ]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

#### Describe data

In [ ]:
print("Train data's size: ", train_data.shape)
print("Test data's size: ", test_data.shape)

In [ ]:
numCols = list(train_data.select_dtypes(exclude='object').columns)

# remove "id" from the category columns
numCols.remove("id")
print(f"There are {len(numCols)} numerical features:\n", numCols)

In [ ]:
catCols = list(train_data.select_dtypes(include='object').columns)

print(f"There are {len(catCols)} categorical features:\n", catCols)

<a name="data-preprocessing"></a>
## Data Preprocessing and Feature Engineering

<a name="feature-add"></a>
## Feature Adder

In [ ]:
class FeatureAdder(BaseEstimator, TransformerMixin):
    """
    A custom transformer that adds new features to the dataset.
    Specifically, it extracts the episode number from the 'Episode_Title' column.
    """
    
    def fit(self, X, y=None):
        """
        Fit method for the transformer. Does nothing as this transformer does not require fitting.
        """
        return self
    
    def transform(self, data):
        """
        Transform method to add new features to the dataset.
        """

        data["DIV_HT_WT"] = data["Height"] / data["Weight"]
        data["DIV_BT_DUR"] = data["Body_Temp"] / data["Duration"]
        data["DIV_HR_DUR"] = data["Heart_Rate"] / data["Duration"]
        data["DIV_BT_HR"] = data["Body_Temp"] / data["Heart_Rate"]

        # add log-transform of skewed features
        skewed_feats = ['Body_Temp', 'Height', 'Duration', 'Heart_Rate']
        for feat in skewed_feats:
            data[f'LOG_{feat}'] = np.log1p(data[feat])
        return data


<a name="modeling"></a>
# Modeling

<a name="random-forest"></a>
## Random Forest

In [ ]:
def train_and_test_model(search_model, X_train, Y_train, X_test):
    """
    Trains the model using GridSearchCV, finds the best model, and makes predictions on the test set.

    Parameters:
    search_model: GridSearchCV object
        The model wrapped in GridSearchCV for hyperparameter tuning.
    X_train: DataFrame
        The training features.
    Y_train: Series
        The target variable for training.
    X_test: DataFrame
        The test features for making predictions.

    Returns:
    best_model: Estimator
        The best model found by GridSearchCV.
    y_pred: ndarray
        Predictions made by the best model on the test set.
    """
    # Fit the model to the training data
    search_model.fit(X_train, Y_train)

    # Print the cross-validation results
    print(search_model.cv_results_)
    
    # Extract the best model from GridSearchCV
    best_model = search_model.best_estimator_
    print(best_model)
    
    # Make predictions on the test set using the best model
    y_pred = best_model.predict(X_test)

    return best_model, y_pred

In [ ]:
# Define the preprocessing pipeline
preprocessor = Pipeline(
    steps=[
        # Add new features using the custom FeatureAdder transformer
        ("feature_add", FeatureAdder()),
        # Apply column transformations: impute missing values and encode categorical features
        ("column_transform", 
         ColumnTransformer(
            transformers=[
                # Label encode categorical columns, ignoring unknown categories
                ("sex_encoder", OrdinalEncoder(), ["Sex"]),
                ('keep', 'passthrough', ["DIV_HT_WT", "DIV_BT_DUR", "DIV_HR_DUR", "DIV_BT_HR",
                                         "LOG_Body_Temp", "LOG_Height", "LOG_Duration", "LOG_Heart_Rate"]), 
            ])
        )
    ]
)

# Define the full pipeline with preprocessing and the Random Forest model
pipeline = Pipeline([
    ("preprocessor", preprocessor),  # Add the preprocessing pipeline
    ("randomforest", RandomForestRegressor(warm_start=True, n_jobs=-1, random_state=22, verbose=3))  # Random Forest Regressor
])

# Define the parameter grid for hyperparameter tuning
param_grid = [
    {
        "randomforest__n_estimators": [100],  # Number of trees in the forest
        "randomforest__max_depth": [50]      # Maximum depth of the tree
    }
]

# Use GridSearchCV for hyperparameter tuning with cross-validation
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=3,  # 3-fold cross-validation
    scoring="neg_mean_squared_log_error",  # Scoring metric
    return_train_score=True, 
    n_jobs=-1,  # Use all available processors
    verbose=3  # Verbosity level
)

In [ ]:
target = "Calories"
_, predictions = train_and_test_model(grid_search, train_data.drop(columns=[target]), train_data[target], test_data)

In [ ]:
# Create a submission DataFrame with the "id" column from the test data
submission_df = pd.DataFrame(test_data["id"])

# Add the predicted target values to the submission DataFrame
submission_df[target] = predictions

# Save the submission DataFrame to a CSV file
submission_df.to_csv("data/randomforest.csv", index=False)